In [1]:
import os
os.environ["ANTHROPIC_API_KEY"] = "paste-your-key-here"
#!pip install anthropic
"""
CYBERARK RISK-ARB POSITION
Modelled as of November 2025
Palo Alto agreed to acquire CyberArk at ~$71/share
CyberArk trading at ~$65 - reflecting deal break risk
contract size assumed to be all options instead of straight shares.
"""

'\nCYBERARK RISK-ARB POSITION\nModelled as of November 2025\nPalo Alto agreed to acquire CyberArk at ~$71/share\nCyberArk trading at ~$65 - reflecting deal break risk\n'

In [2]:
from scipy.stats import norm
import numpy as np

def BS_greeks(S, K, T, r, sigma, option_type='call'):
    """
    S = current spot price
    K = strike price
    T = time to expiry in years
    r = risk free rate
    sigma = implied volatility (e.g. 0.3 = 30%)
    option_type = 'call' or 'put'
    """
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    if option_type == 'call':
        delta = norm.cdf(d1)
    else:
        delta = norm.cdf(d1) - 1

    gamma = norm.pdf(d1) / (S * sigma * np.sqrt(T))
    vega  = S * norm.pdf(d1) * np.sqrt(T) * 0.01  # per 1% vol move
    theta = (-S * norm.pdf(d1) * sigma / (2 * np.sqrt(T))) / 365  # per day

    return {
        "delta": round(delta, 4),
        "gamma": round(gamma, 4),
        "vega":  round(vega, 4),
        "theta": round(theta, 4)
    }

print("Black-Scholes Greeks function loaded")

Black-Scholes Greeks function loaded


In [4]:
from datetime import datetime

spot_price      = 65.0        # CyberArk price @ Nov 2025
strike_price    = 71.0        # Palo Alto deal price
expiry_date     = "2026-03-31"  # expected deal close
implied_vol     = 0.25        # low vol chosen, deal largely expected to close
risk_free_rate  = 0.05        
option_type     = "call"      
position_size   = 15000       # Maven's actual position size
direction       = "long"      

today = datetime(2025, 11, 20)  # fixing date to Maven's filing date
expiry = datetime.strptime(expiry_date, "%Y-%m-%d")
T = (expiry - today).days / 252

print(f"Position: {direction} {position_size} {option_type} options")

Position: long 15000 call options


In [12]:
# Calc Greeks
greeks = BS_greeks(
    S           = spot_price,
    K           = strike_price,
    T           = T,
    r           = risk_free_rate,
    sigma       = implied_vol,
    option_type = option_type
)

# Adjust for direction
multiplier = 1 if direction == "long" else -1

print("=== GREEKS ===")
print(f"Delta : {round(greeks['delta'] * multiplier, 4)}")
print(f"Gamma : {round(greeks['gamma'] * multiplier, 4)}")
print(f"Vega  : {round(greeks['vega']  * multiplier, 4)}")
print(f"Theta : {round(greeks['theta'] * multiplier, 4)}")

=== GREEKS ===
Delta : 0.3992
Gamma : 0.033
Vega  : 0.181
Theta : -0.0119


In [17]:
import anthropic
import json

def generate_risk_report(position_details, greeks):
    
    client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
    
    context = f"""
    You are a risk analyst assistant at a proprietary trading firm.
    
    You are assessing this position AS OF NOVEMBER 2025 ONLY.
    You do not know the outcome of this deal after this date.
    Do not reference any future outcomes.
    
    A trader has proposed the following risk-arb position:
    - Asset: CyberArk Software (CYBR) call options
    - Context: Palo Alto Networks announced acquisition of CyberArk at $71/share. Deal break risk exists.
    - The strike represents the deal price target.
    - Direction: {position_details['direction']}
    - Option type: {position_details['option_type']}
    - Position size: {position_details['size']} contracts
    - Spot price: ${position_details['spot']}
    - Strike price: ${position_details['strike']}
    - Expiry: {position_details['expiry']}
    - Implied volatility: {position_details['vol']*100}%
    
    The computed Greeks are:
    - Delta: {greeks['delta']}
    - Gamma: {greeks['gamma']}
    - Vega: {greeks['vega']}
    - Theta: {greeks['theta']}
    
    Write a concise plain English risk summary for the risk manager.
    Cover: deal break risk, directional risk, vol risk, time decay, and key concerns.
    Do not recompute or second guess the numbers above.
    Write in bullet points. One bullet per key risk factor.
    """
    
    message = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=1000,
        temperature=0,
        messages=[{"role": "user", "content": context}]
    )
    
    return message.content[0].text

print("Risk report function loaded")

Risk report function loaded


In [18]:
# Package position details into a dictionary
position_details = {
    'direction':   direction,
    'option_type': option_type,
    'size':        position_size,
    'spot':        spot_price,
    'strike':      strike_price,
    'expiry':      expiry_date,
    'vol':         implied_vol
}

# Generate the risk report
print("=== LLM RISK REPORT ===")
print("Generating report...\n")

report = generate_risk_report(position_details, greeks)
print(report)

print("\n=== RAW GREEKS ===")
print(f"Delta : {greeks['delta']}")
print(f"Gamma : {greeks['gamma']}")
print(f"Vega  : {greeks['vega']}")
print(f"Theta : {greeks['theta']}")

=== LLM RISK REPORT ===
Generating report...

**Risk Summary: CYBR Call Options Position (15,000 contracts)**

• **Deal Break Risk**: Primary concern - if the Palo Alto acquisition falls through, CYBR stock will likely drop significantly below current $65 level, making these $71 strike calls worthless; this represents the largest potential loss scenario

• **Directional Risk**: Position needs CYBR to move up $6 (9.2%) from current $65 to reach $71 strike for intrinsic value; delta of 0.40 means position gains/loses ~$240,000 per $1 move in underlying stock

• **Volatility Risk**: High vega exposure of $271,500 per 1% vol change; if deal certainty increases and implied vol drops from 25%, position loses value even if stock stays flat

• **Time Decay**: Bleeding $1,785 per day in theta; with 4+ months to March expiry, time decay will accelerate as expiration approaches, especially problematic if deal timeline extends

• **Key Concerns**: Position is essentially a binary bet on deal compl